# Description

In this notebook, I will load the generated code from HumanEval (by GPT4) and get the code embedding

In [1]:
import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch 
import transformers
from transformers import AutoTokenizer, AutoModel

from utils.evaluation_human_eval import *

In [2]:
print(torch.__version__)    
print(torch.version.cuda)
print(torch.cuda.nccl.version())
print(transformers.__version__)

2.6.0+cu124
12.4
(2, 21, 5)
4.45.2


# 1. Load code embedding

In [3]:
# Load model + tokenizer
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [4]:
def embed_code(code_snippet: str):
    # Tokenize input
    inputs = tokenizer(code_snippet, return_tensors="pt", truncation=True, padding=True)
    
    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Use CLS token embedding as the vector
    embeddings = outputs.last_hidden_state[:, 0, :]
    return embeddings.squeeze().numpy()

In [5]:
code = """
def add(a, b):
    return a + b
"""

vec = embed_code(code)
print("Embedding shape:", vec.shape)

Embedding shape: (768,)


# 2. Generate Embeding for Human Eval dataset

In [6]:
PATH_CSV_HUMAN_EVAL_GPT = "data/generated/human_eval_generated_gpt4o.csv"
PATH_CSV_HUMAN_EVAL_DEEPSEEK = "data/generated/human_eval_generated_deepseek.csv"
PATH_CSV_HUMAN_EVAL_LLAMA = "data/generated/human_eval_generated_llama.csv"
PATH_CSV_HUMAN_EVAL_QWEN = "data/generated/human_eval_generated_qwen.csv"

In [7]:
df_deepseek = pd.read_csv(PATH_CSV_HUMAN_EVAL_DEEPSEEK)
df_llama = pd.read_csv(PATH_CSV_HUMAN_EVAL_LLAMA)
df_qwen = pd.read_csv(PATH_CSV_HUMAN_EVAL_QWEN)
df_gpt = pd.read_csv(PATH_CSV_HUMAN_EVAL_GPT)


In [8]:
# Combine all dataframes with an additional column to indicate the source model
df_deepseek['model'] = 'deepseek'
df_llama['model'] = 'llama'
df_qwen['model'] = 'qwen'
df_gpt['model'] = 'gpt4o'

df = pd.concat([df_deepseek, df_llama, df_qwen, df_gpt], ignore_index=True)
df.drop_duplicates(subset=['generated_code', 'test_case'], inplace=True)
df = df.reset_index(drop=True)
print("Combined dataframe shape:", df.shape)

Combined dataframe shape: (628, 8)


In [9]:
df['generated_code_embedd'] = df['generated_code'].apply(embed_code)
df['test_case_embedd'] = df['test_case'].apply(embed_code)

print(f"[INFO] Embedding generation completed!")

[INFO] Embedding generation completed!


## 2.1. Test embedding space

In [10]:
def distance_2_vector(vec1, vec2, type="l2"):
    if type == "l2":
        return np.linalg.norm(vec1 - vec2)
    elif type == "cosine":
        return 1 - np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    else:
        raise ValueError("Unknown distance type")

In [11]:
idx = np.random.randint(0, len(df))

gen_code = df.loc[idx, 'generated_code']
test_case = df.loc[idx, 'test_case']
entry_point = df.loc[idx, 'entry_point']

gen_code_embedd = df.loc[idx, 'generated_code_embedd']
test_case_embedd = df.loc[idx, 'test_case_embedd']

print("Generated Code:\n", gen_code)
print("-"*50)
print("Test Case:\n", test_case)

Generated Code:
 def digits(n):
    product = 1
    for digit in str(n):
        if int(digit) % 2 != 0:
            product *= int(digit)
    return product
--------------------------------------------------
Test Case:
 def check(candidate):

    # Check some simple cases
    assert candidate(5) == 5
    assert candidate(54) == 5
    assert candidate(120) ==1
    assert candidate(5014) == 5
    assert candidate(98765) == 315
    assert candidate(5576543) == 2625

    # Check some edge cases that are easy to work out by hand.
    assert candidate(2468) == 0




In [12]:
result = evaluate_asserts(gen_code, test_case, entry_point)
print(result)

{'total_asserts': 7, 'passed': 6, 'percentage': 0.8571428571428571, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('fail', 'AssertionError()')]}


In [13]:
distance = distance_2_vector(gen_code_embedd, test_case_embedd, type="l2")
print(f"L2 Distance between generated code and test case embeddings: {distance}")

L2 Distance between generated code and test case embeddings: 3.656877040863037


## Save file

In [14]:
OUTPUT_CSV_PATH = "data/embedding/human_eval_codebert_embeddings.csv"
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"[INFO] Saved embeddings to {OUTPUT_CSV_PATH}")

[INFO] Saved embeddings to data/embedding/human_eval_codebert_embeddings.csv
